# # PRE_11 — Vuelos domésticos 2006-2008
#
# **Qué se hace:** con los vuelos nacionales agregados por aerolínea (por día/hora y por mes) se
# construyen KPIs de puntualidad, un ranking de aerolíneas ajustado por horario, el mapa día x hora
# de demoras, la estacionalidad y segmentos prioritarios.
#
# **Definiciones**
# - *Tasa de demora* = vuelos con salida demorada >= 15 min / vuelos operados.
# - *Tasa de cancelación* = cancelados / programados.
# - *Ajuste por horario* (estandarización indirecta, como en LAB_11): demoras esperadas de una aerolínea =
#   suma sobre horas de (sus vuelos operados x tasa nacional de esa hora). Razón observado/esperado > 1
#   significa que se demora más de lo que explican sus horarios.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Funciona tanto desde la carpeta de la actividad como desde notebooks/ (Jupyter)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
SUBMISSION = ROOT / "submission"
SUBMISSION.mkdir(exist_ok=True)

dh = pd.read_csv(ROOT / "data" / "flights_by_carrier_day_hour.csv.gz")
mo = pd.read_csv(ROOT / "data" / "flights_by_carrier_month.csv.gz")
print(dh.shape, mo.shape)

(96071, 10) (707, 8)


# ## 1. Calidad de datos

In [2]:
cols = ["scheduled_flights", "cancelled_flights", "operated_flights", "delayed_departure_15_flights",
        "positive_departure_delay_minutes"]
for name, df, key in [("día/hora", dh, ["year", "month", "day_of_week", "scheduled_departure_hour", "reporting_airline"]),
                      ("mes", mo, ["year", "month", "reporting_airline"])]:
    print(f"[{name}] nulos: {int(df.isna().sum().sum())} | llaves duplicadas: {int(df.duplicated(key).sum())} | "
          f"programados != cancelados + operados: {int(((df.scheduled_flights - df.cancelled_flights - df.operated_flights).abs() > 0).sum())} | "
          f"demorados > operados: {int((df.delayed_departure_15_flights > df.operated_flights).sum())} | "
          f"negativos: {int((df[cols] < 0).sum().sum())}")
print("Años:", sorted(dh.year.unique()), "| aerolíneas:", dh.reporting_airline.nunique(),
      "| horas:", dh.scheduled_departure_hour.min(), "-", dh.scheduled_departure_hour.max())
# Los dos archivos deben contar la misma historia
tot_dh, tot_mo = dh[cols].sum(), mo[cols].sum()
assert (tot_dh == tot_mo).all(), "los totales de los dos archivos no coinciden"
print(tot_dh.astype(int).to_string())

[día/hora] nulos: 0 | llaves duplicadas: 0 | programados != cancelados + operados: 0 | demorados > operados: 0 | negativos: 0
[mes] nulos: 0 | llaves duplicadas: 0 | programados != cancelados + operados: 0 | demorados > operados: 0 | negativos: 0
Años: [2006, 2007, 2008] | aerolíneas: 21 | horas: 0 - 23
scheduled_flights                    21607106
cancelled_flights                      420175
operated_flights                     21186931
delayed_departure_15_flights          4324953
positive_departure_delay_minutes    270951248


# **Conclusión:** datos completos y consistentes (sin nulos ni duplicados; programados = cancelados + operados;
# los totales de ambos archivos coinciden exactamente). Los conteos vienen como `float`, se tratan como enteros al sumar.

In [3]:
def rates(g):
    """Agrega conteos y calcula tasas sobre una tabla ya agrupada."""
    g["cancel_rate"] = (g.cancelled_flights / g.scheduled_flights).round(4)
    g["delay_rate"] = (g.delayed_departure_15_flights / g.operated_flights).round(4)
    g["avg_delay_min"] = (g.positive_departure_delay_minutes / g.operated_flights).round(2)
    return g


def agrupar(df, by):
    g = df.groupby(by)[cols].sum()
    g[cols[:-1]] = g[cols[:-1]].astype(int)
    return rates(g).reset_index()

# ## 2. KPIs globales (`overall_kpis.csv`)

In [4]:
tot = dh[cols].sum()
national_rate = tot.delayed_departure_15_flights / tot.operated_flights
overall = pd.DataFrame([
    ("period", f"{dh.year.min()}-{dh.year.max()}"),
    ("carriers", dh.reporting_airline.nunique()),
    ("scheduled_flights", int(tot.scheduled_flights)),
    ("cancelled_flights", int(tot.cancelled_flights)),
    ("operated_flights", int(tot.operated_flights)),
    ("delayed_departure_15_flights", int(tot.delayed_departure_15_flights)),
    ("cancel_rate", round(tot.cancelled_flights / tot.scheduled_flights, 4)),
    ("delay_rate", round(national_rate, 4)),
    ("avg_delay_min_per_operated_flight", round(tot.positive_departure_delay_minutes / tot.operated_flights, 2)),
    ("avg_delay_min_per_delayed_flight_upper_bound", round(tot.positive_departure_delay_minutes / tot.delayed_departure_15_flights, 2)),
], columns=["metric", "value"])
overall.to_csv(SUBMISSION / "overall_kpis.csv", index=False)
overall

,metric,value
0,period,2006-2008
1,carriers,21
2,scheduled_flights,21607106
3,cancelled_flights,420175
4,operated_flights,21186931
5,delayed_departure_15_flights,4324953
6,cancel_rate,0.0194
7,delay_rate,0.2041
8,avg_delay_min_per_operated_flight,12.79
9,avg_delay_min_per_delayed_flight_upper_bound,62.65


# *Nota:* los minutos de demora positiva incluyen también salidas con 1-14 min de retraso, por eso el promedio
# por vuelo demorado >= 15 min es una cota superior de referencia, no un promedio exacto.

# ## 3. Aerolíneas con ajuste por horario (`carrier_summary.csv`)
# Solo se ranquean aerolíneas con al menos 100.000 vuelos operados (criterio de LAB_11); el resto se informa sin ranking.

In [5]:
hourly = dh.groupby("scheduled_departure_hour")[["operated_flights", "delayed_departure_15_flights"]].sum()
hourly["hour_rate"] = hourly.delayed_departure_15_flights / hourly.operated_flights

carrier = agrupar(dh, "reporting_airline")
by_ch = dh.groupby(["reporting_airline", "scheduled_departure_hour"]).operated_flights.sum().reset_index()
by_ch["expected"] = by_ch.operated_flights * by_ch.scheduled_departure_hour.map(hourly.hour_rate)
carrier["expected_delayed_flights"] = carrier.reporting_airline.map(by_ch.groupby("reporting_airline").expected.sum()).round(1)
carrier["observed_to_expected_ratio"] = (carrier.delayed_departure_15_flights / carrier.expected_delayed_flights).round(4)
carrier["meets_min_volume"] = carrier.operated_flights >= 100_000
q = carrier[carrier.meets_min_volume]
carrier["crude_rank"] = q.delay_rate.rank(ascending=False, method="min").reindex(carrier.index).astype("Int64")
carrier["adjusted_rank"] = q.observed_to_expected_ratio.rank(ascending=False, method="min").reindex(carrier.index).astype("Int64")
carrier["rank_change"] = carrier.crude_rank - carrier.adjusted_rank  # > 0: el ajuste la empeora
carrier["share_of_flights"] = (carrier.operated_flights / tot.operated_flights).round(4)
carrier = carrier.sort_values(["meets_min_volume", "adjusted_rank", "operated_flights"], ascending=[False, True, False])
carrier.to_csv(SUBMISSION / "carrier_summary.csv", index=False)
carrier

,reporting_airline,scheduled_flights,cancelled_flights,operated_flights,delayed_departure_15_flights,positive_departure_delay_minutes,cancel_rate,delay_rate,avg_delay_min,expected_delayed_flights,observed_to_expected_ratio,meets_min_volume,crude_rank,adjusted_rank,rank_change,share_of_flights
6,EV,839952,20729,819223,230689,14688403.0,0.0247,0.2816,17.93,165686.3,1.3923,True,1,1,0,0.0387
11,MQ,1581275,61113,1520162,352450,21663193.0,0.0386,0.2319,14.25,306006.7,1.1518,True,2,2,0,0.0717
1,AA,1882339,45491,1836848,421120,27479762.0,0.0242,0.2293,14.96,371425.6,1.1338,True,3,3,0,0.0867
16,UA,1439525,32708,1406817,317803,21920942.0,0.0227,0.2259,15.58,282334.3,1.1256,True,4,4,0,0.0664
13,OH,711738,22249,689489,150917,8919019.0,0.0313,0.2189,12.94,140965.7,1.0706,True,7,5,2,0.0325
3,B6,543273,7574,535699,118591,8578495.0,0.0139,0.2214,16.01,110787.2,1.0704,True,5,6,-1,0.0253
20,YV,854056,30050,824006,181836,12992153.0,0.0352,0.2207,15.77,170542.7,1.0662,True,6,7,-1,0.0389
4,CO,930995,8302,922693,194198,13108379.0,0.0089,0.2105,14.21,184758.0,1.0511,True,8,8,0,0.0436
2,AS,470691,7132,463559,96411,5761876.0,0.0152,0.2080,12.43,95953.3,1.0048,True,10,9,1,0.0219
18,WN,3469946,31333,3438613,719230,38962701.0,0.0090,0.2092,11.33,716251.6,1.0042,True,9,10,-1,0.1623


# ## 4. Mapa día de semana x hora (`day_hour_delay.csv`)
# `day_of_week`: 1 = lunes ... 7 = domingo (según el archivo).

In [6]:
day_hour = agrupar(dh, ["day_of_week", "scheduled_departure_hour"])
day_hour.to_csv(SUBMISSION / "day_hour_delay.csv", index=False)
print(day_hour.shape)
day_hour.pivot(index="day_of_week", columns="scheduled_departure_hour", values="delay_rate").round(2)

(168, 10)


scheduled_departure_hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
day_of_week,,,,,,,,,,,,,,,,,,,,,
1,0.18,0.17,0.16,0.23,0.17,0.06,0.07,0.09,0.12,0.15,...,0.25,0.26,0.27,0.28,0.29,0.29,0.30,0.27,0.22,0.21
2,0.16,0.15,0.13,0.22,0.12,0.05,0.06,0.08,0.10,0.13,...,0.21,0.22,0.24,0.25,0.26,0.26,0.28,0.25,0.20,0.20
3,0.14,0.14,0.09,0.24,0.10,0.05,0.06,0.08,0.10,0.13,...,0.22,0.24,0.25,0.27,0.28,0.28,0.30,0.26,0.21,0.21
4,0.16,0.15,0.11,0.27,0.10,0.05,0.06,0.08,0.11,0.13,...,0.25,0.27,0.29,0.31,0.33,0.34,0.36,0.33,0.27,0.27
5,0.18,0.19,0.14,0.28,0.11,0.06,0.07,0.09,0.12,0.15,...,0.28,0.30,0.32,0.34,0.35,0.36,0.38,0.34,0.28,0.28
6,0.18,0.19,0.15,0.30,0.15,0.07,0.07,0.09,0.11,0.14,...,0.22,0.23,0.23,0.24,0.24,0.24,0.25,0.22,0.18,0.19
7,0.13,0.16,0.11,0.19,0.14,0.07,0.06,0.08,0.10,0.12,...,0.24,0.26,0.28,0.30,0.31,0.33,0.35,0.32,0.26,0.25


# ## 5. KPIs mensuales nacionales y estacionalidad

In [7]:
monthly = agrupar(mo, ["year", "month"])
monthly.to_csv(SUBMISSION / "monthly_national_kpis.csv", index=False)

# Estacionalidad: mes del año con las tasas agrupadas de los tres años; índice 1.00 = promedio nacional
season = agrupar(mo, "month")
season["delay_rate_index"] = (season.delay_rate / national_rate).round(3)
season["cancel_rate_index"] = (season.cancel_rate / (tot.cancelled_flights / tot.scheduled_flights)).round(3)
season["delay_rank"] = season.delay_rate.rank(ascending=False, method="min").astype(int)
season.to_csv(SUBMISSION / "seasonality.csv", index=False)
season

,month,scheduled_flights,cancelled_flights,operated_flights,delayed_departure_15_flights,positive_departure_delay_minutes,cancel_rate,delay_rate,avg_delay_min,delay_rate_index,cancel_rate_index,delay_rank
0,1,1808611,42872,1765739,357042,21736250.0,0.0237,0.2022,12.31,0.991,1.219,7
1,2,1666087,57354,1608733,375533,23197856.0,0.0344,0.2334,14.42,1.143,1.769,3
2,3,1860516,40646,1819870,398489,24529350.0,0.0218,0.2190,13.48,1.073,1.121,5
3,4,1798125,28097,1770028,327624,19836400.0,0.0156,0.1851,11.21,0.907,0.802,8
4,5,1840821,20127,1820694,320945,19249504.0,0.0109,0.1763,10.57,0.864,0.561,9
5,6,1836260,38262,1797998,441292,29096138.0,0.0208,0.2454,16.18,1.202,1.070,2
6,7,1897735,34839,1862896,428198,28281346.0,0.0184,0.2299,15.18,1.126,0.946,4
7,8,1894290,31913,1862377,389281,24804682.0,0.0168,0.2090,13.32,1.024,0.864,6
8,9,1726032,26370,1699662,253335,15639390.0,0.0153,0.1491,9.20,0.730,0.787,12
9,10,1797915,21975,1775940,295951,18033170.0,0.0122,0.1666,10.15,0.816,0.627,10


# ## 6. Segmentos prioritarios (`priority_segments.csv`)
# Segmento = aerolínea x franja horaria (madrugada 0-5, mañana 6-11, tarde 12-17, noche 18-23).
# Se compara contra lo esperado según la tasa nacional de cada hora; `excess_delayed_flights` = observadas - esperadas.
# Se consideran segmentos con al menos 10.000 vuelos operados. Prioridad **alta** si la razón >= 1,10,
# **media** si está entre 1,00 y 1,10, **baja** si es menor que 1. Orden: mayor exceso de demoras primero.

In [8]:
bins = pd.cut(dh.scheduled_departure_hour, [-1, 5, 11, 17, 23], labels=["madrugada", "mañana", "tarde", "noche"])
seg = dh.assign(time_block=bins.astype(str))
seg["expected"] = seg.operated_flights * seg.scheduled_departure_hour.map(hourly.hour_rate)
sg = seg.groupby(["reporting_airline", "time_block"]).agg(
    operated_flights=("operated_flights", "sum"),
    delayed_departure_15_flights=("delayed_departure_15_flights", "sum"),
    expected_delayed_flights=("expected", "sum"),
).reset_index()
sg = sg[sg.operated_flights >= 10_000].copy()
sg[["operated_flights", "delayed_departure_15_flights"]] = sg[["operated_flights", "delayed_departure_15_flights"]].astype(int)
sg["delay_rate"] = (sg.delayed_departure_15_flights / sg.operated_flights).round(4)
sg["observed_to_expected_ratio"] = (sg.delayed_departure_15_flights / sg.expected_delayed_flights).round(4)
sg["excess_delayed_flights"] = (sg.delayed_departure_15_flights - sg.expected_delayed_flights).round(1)
sg["expected_delayed_flights"] = sg.expected_delayed_flights.round(1)
sg["priority"] = np.select([sg.observed_to_expected_ratio >= 1.10, sg.observed_to_expected_ratio >= 1.0],
                           ["alta", "media"], default="baja")
sg = sg.sort_values("excess_delayed_flights", ascending=False).reset_index(drop=True)
sg.to_csv(SUBMISSION / "priority_segments.csv", index=False)
sg.head(10)

,reporting_airline,time_block,operated_flights,delayed_departure_15_flights,expected_delayed_flights,delay_rate,observed_to_expected_ratio,excess_delayed_flights,priority
0,EV,tarde,335518,113989,81691.3,0.3397,1.3954,32297.7,alta
1,WN,noche,769249,261336,233101.7,0.3397,1.1211,28234.3,alta
2,MQ,mañana,628178,100939,76037.8,0.1607,1.3275,24901.2,alta
3,AA,tarde,723064,200129,175615.7,0.2768,1.1396,24513.3,alta
4,EV,mañana,334259,62203,41367.2,0.1861,1.5037,20835.8,alta
5,AA,noche,350484,124098,104119.0,0.3541,1.1919,19979.0,alta
6,MQ,tarde,597582,162037,145484.8,0.2712,1.1138,16552.2,alta
7,WN,tarde,1295508,332952,316412.4,0.2570,1.0523,16539.6,media
8,UA,tarde,509911,139300,124667.3,0.2732,1.1174,14632.7,alta
9,UA,mañana,598632,84093,71181.3,0.1405,1.1814,12911.7,alta


# ## 7. Verificación de consistencia

In [9]:
assert carrier.operated_flights.sum() == day_hour.operated_flights.sum() == monthly.operated_flights.sum() == int(tot.operated_flights)
assert season.operated_flights.sum() == int(tot.operated_flights)
assert len(day_hour) == 7 * 24 and len(monthly) == 36 and len(season) == 12
# la suma de esperadas por aerolínea debe igualar las demoras totales (propiedad de la estandarización indirecta)
assert abs(carrier.expected_delayed_flights.sum() - tot.delayed_departure_15_flights) < 5
for name in ["carrier_summary", "day_hour_delay", "monthly_national_kpis", "overall_kpis", "priority_segments", "seasonality"]:
    print(f"{name}.csv", len(pd.read_csv(SUBMISSION / f"{name}.csv")), "filas")

carrier_summary.csv 21 filas
day_hour_delay.csv 168 filas
monthly_national_kpis.csv 36 filas
overall_kpis.csv 10 filas
priority_segments.csv 68 filas
seasonality.csv 12 filas


# ## Hallazgos

In [10]:
q = carrier[carrier.meets_min_volume].sort_values("adjusted_rank")
print(f"Tasa nacional de demora: {national_rate:.1%} | cancelación: {tot.cancelled_flights / tot.scheduled_flights:.1%}")
print(q[["reporting_airline", "delay_rate", "observed_to_expected_ratio", "crude_rank", "adjusted_rank", "rank_change"]].to_string(index=False))
print("\nMovimientos de ranking:", q[q.rank_change != 0][["reporting_airline", "rank_change"]].values.tolist())
dh_peak = day_hour.loc[day_hour[day_hour.operated_flights > 5000].delay_rate.idxmax()]
print(f"Peor celda día x hora (>5.000 vuelos): día {int(dh_peak.day_of_week)}, hora {int(dh_peak.scheduled_departure_hour)}, {dh_peak.delay_rate:.1%}")
print("Meses más demorados:", season.sort_values("delay_rate", ascending=False).head(3)[["month", "delay_rate"]].values.tolist())
print("Segmentos prioridad alta:", int((sg.priority == "alta").sum()), "de", len(sg))

Tasa nacional de demora: 20.4% | cancelación: 1.9%
reporting_airline  delay_rate  observed_to_expected_ratio  crude_rank  adjusted_rank  rank_change
               EV      0.2816                      1.3923           1              1            0
               MQ      0.2319                      1.1518           2              2            0
               AA      0.2293                      1.1338           3              3            0
               UA      0.2259                      1.1256           4              4            0
               OH      0.2189                      1.0706           7              5            2
               B6      0.2214                      1.0704           5              6           -1
               YV      0.2207                      1.0662           6              7           -1
               CO      0.2105                      1.0511           8              8            0
               AS      0.2080                      1.0048          

# - Tasa nacional de demora 20,4 % y cancelación 1,9 % (21,2 M de vuelos operados). EV es la peor aerolínea
#   en ambos rankings (28,2 % crudo; 1,39x lo esperado); HA es la mejor (4,8 %).
# - El ajuste mueve a 8 de 19 aerolíneas, siempre 1-2 posiciones (p. ej. OH sube de 7.º a 5.º y F9 baja de 15.º a 17.º):
#   el ranking crudo cambia poco aquí, pero el ajuste corrige las posiciones intermedias.
# - Las demoras son máximas el viernes hacia las 20 h (~38 %); diciembre, junio y febrero son los meses más demorados.
# - La demora crece a lo largo del día (se acumula): una aerolínea con más vuelos de tarde y noche parece peor
#   sin serlo. El ranking ajustado (observado/esperado) puede diferir del ranking crudo; las filas con
#   `rank_change != 0` son las que el ranking sin ajustar juzga mal por sus horarios.
# - La estacionalidad (`seasonality.csv`) muestra los meses críticos; el índice > 1 marca meses peores que el promedio.
# - `priority_segments.csv` apunta dónde hay más demoras *por encima de lo esperado*: es la lista de trabajo
#   para mejorar puntualidad, ordenada por impacto en vuelos y no solo por tasa.